<a href="https://colab.research.google.com/github/moshfikaurme-dot/paython-practice/blob/main/grapevine_leaves_image_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Requirements

In [ ]:
import matplotlib.pyplot as plt
import warnings
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Conv2D, Flatten, Input, MaxPooling2D, Dropout, BatchNormalization, Reshape
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import accuracy_score
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.applications import VGG19
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.preprocessing import LabelEncoder , OneHotEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping,CSVLogger
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore") #instruct Python to ignore all warning messages. This means that any warnings that would normally be printed to the console are suppressed.

#Import Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
# Define the paths to your zip files
# rooms_dataset = '/content/drive/MyDrive/rooms_dataset'


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
dataset_path= '/content/drive/MyDrive/Grapevine_Leaves_Image_Dataset'
lists = os.listdir(dataset_path)
print("Types of folder found: ", len(dataset_path))
print(lists)

In [ ]:
flags = [] #This initializes an empty list called rooms that will store tuples of (room_type, file_path).

for item in lists:
 # Get all the file names
 all_flags = os.listdir(dataset_path + '/' +item)
 #print(all_shoes)

 # Add them to the list
 for flag in all_flags:
    flags.append((item, str(dataset_path + '/' +item) + '/' + flag))
       #eta change kore dekhte hobe

print(flags[2:6])

In [ ]:
import cv2
path = '/content/drive/MyDrive/Grapevine_Leaves_Image_Dataset'

im_size = 224
images = []
labels = []

for i in lists:
    data_path = path + '/' + str(i)
    filenames = [i for i in os.listdir(data_path) ]
    print(filenames)
    for f in filenames:
        img = cv2.imread(data_path + '/' + f)
        img = cv2.resize(img, (im_size, im_size))
        images.append(img)
        labels.append(i)

print(len(images))
print(len(labels))

#Build a dataframe

In [ ]:
# Build a dataframe    --     flags = []     #This initializes an empty list called rooms that will store tuples of (flag_type, file_path).

flags_df = pd.DataFrame(data=flags, columns=['flag type', 'image'])
print(flags_df.head())  #1st kichu part
print(flags_df.tail())  #last kichu part

In [ ]:
print("Total number of leafs in the dataset: ", len(flags_df))

In [ ]:
flag_count = flags_df['flag type'].value_counts()  #flag type er moddhe value count korse

print("leaf in each category: ")
print(flag_count)

In [ ]:
images = np.array(images)  #This converts the images list into a NumPy array
images.shape       # retrieves the dimensions of the array

In [ ]:
images = images.astype('float32') / 255.0  #astype hocche changing the data type    -- Changing Data Type and Normalizing  - : This converts the data type of the images array to float32
#Dividing by 255.0 scales the pixel values from the range [0, 255] to [0, 1]
images.shape         # retrieves the dimensions of the array

In [ ]:
y = flags_df['flag type'].values   #.values: This converts the selected column into a NumPy array. The resulting array, y, contains the values from the 'flag type' column.
print(y)

In [ ]:
y_labelencoder = LabelEncoder ()  #convert categorical labels into numeric form
y = y_labelencoder.fit_transform (y)
print(len(y))
print (y)  #count the length too   ['Ak'   0 , 'Ala_Idris'  1, 'Buzgulu'  2, 'Nazli'  4, 'Dimnit' 3

#**Limitation of label Encoding
If the encoded values imply a relationship (e.g., Red = 0 and Blue = 2 might suggest Red < Blue), the model may incorrectly interpret the data as ordinal. To address this, we consider using One-Hot Encoding.**

In [ ]:
y = y.reshape(-1,1)
#This reshapes the array y into a 2-dimensional array with one column.
#The -1 means "infer the appropriate number of rows," so the resulting shape will be (number_of_samples, 1).
#This is necessary because OneHotEncoder expects the input to be in a 2-dimensional format.
# print(y)    --- ek line a shob chole ashe, tai array te convert kora hoyehe
onehotencoder = OneHotEncoder()  #transforms categorical data into a binary matrix,
Y = onehotencoder.fit_transform(y).toarray()
print(Y)   #print Y
print(Y.shape)

#Train Test Val split

In [ ]:

# Shuffle the data
images, Y = shuffle(images, Y, random_state=1)

# First split: Separate 20% for testing
train_val_x, test_x, train_val_y, test_y = train_test_split(images, Y, test_size=0.20, random_state=415, stratify=Y)
#This splits the dataset into:
# Training + Validation (80%): train_val_x, train_val_y.
# Testing (20%): test_x, test_y.

# Second split: Separate 10% for validation from the remaining 80%
train_x, val_x, train_y, val_y = train_test_split(train_val_x, train_val_y, test_size=0.125, random_state=415, stratify=train_val_y)
# Note: test_size=0.125 because 10% of the original dataset is 12.5% of the remaining 80%.
# This splits the training + validation (80%) set into:

# Training (70%): train_x, train_y.
# Validation (10%): val_x, val_y.

# Why test_size=0.125?

# 10% of the original dataset is 12.5% of the remaining 80% (since
# 10/80= 0.125


# Inspect the shapes of the datasets
print("Training data shape:", train_x.shape)
print("Training labels shape:", train_y.shape)
print("Validation data shape:", val_x.shape)
print("Validation labels shape:", val_y.shape)
print("Testing data shape:", test_x.shape)
print("Testing labels shape:", test_y.shape)

#just train test spli

In [ ]:
images, Y = shuffle(images, Y, random_state=1) #Ensures reproducibility by fixing the random seed, so the shuffling produces the same result every time the code is run.

train_x, test_x, train_y, test_y = train_test_split(images, Y, test_size=0.05, random_state=415, stratify= Y) #The number 415 in the random_state parameter doesn't have any inherent importance beyond serving as a seed for the random number generator. It's just an arbitrary value chosen to ensure reproducibility of your code.
#test_size=0.05:  This specifies that 5% of the data should be allocated to the testing set, and the remaining 95% will be used for training.

#inpect the shape of the training and testing.
print(train_x.shape)
print(train_y.shape)
print(test_x.shape)
print(test_y.shape)

#VGG19

In [ ]:
def VGGupdated(input_tensor=None,classes=5):

    img_rows, img_cols = 224, 224
    img_channels = 3

    img_dim = (img_rows, img_cols, img_channels)

    img_input = Input(shape=img_dim)

    # Block 1
    x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv1')(img_input)#1
    x = Conv2D(64, (3, 3), activation='relu', padding='same', name='block1_conv2')(x)#2
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

    # Block 2
    x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv1')(x)#3
    x = Conv2D(128, (3, 3), activation='relu', padding='same', name='block2_conv2')(x)#4
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

    # Block 3
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv1')(x)#5
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv2')(x)#6
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv3')(x)#7
    x = Conv2D(256, (3, 3), activation='relu', padding='same', name='block3_conv4')(x)#8
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

    # Block 4
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv1')(x)#9
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv2')(x)#10
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv3')(x)#11
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block4_conv4')(x)#12
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)

    # Block 5
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv1')(x)#13
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv2')(x)#14
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv3')(x)#15
    x = Conv2D(512, (3, 3), activation='relu', padding='same', name='block5_conv4')(x)#16
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block5_pool')(x)


    # Classification block
    x = Flatten(name='flatten')(x) #If the input is a 2D array of shape (rows, columns), it is reshaped into a 1D array of shape (rows * columns). This is necessary before feeding the data into fully connected (dense) layers, which expect 1D inputs.
    x = Dense(4096, activation='relu', name='fc1')(x)#17
    x = Dense(4096, activation='relu', name='fc2')(x)#18
    x = Dense(1000, activation='relu', name='fc3')(x)#19
    x = Dense(classes, activation='softmax', name='predictions')(x)

    # Create model.


    model = Model(inputs = img_input, outputs = x, name='VGGdemo')


    return model

In [ ]:
model = VGGupdated(classes = 5)

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# ================= DRIVE CHECKPOINT + AUTO-RESUME =================
# All checkpoints, epoch logs, best models and results are saved in Google Drive.
import os, json
import pandas as pd
import numpy as np
from pathlib import Path
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, CSVLogger

RESULT_DIR = '/content/drive/MyDrive/Grapevine_VGG19_Results'
os.makedirs(RESULT_DIR, exist_ok=True)

class SaveLastEpoch(Callback):
    """Save the complete model (including optimizer state) after every completed epoch."""
    def __init__(self, checkpoint_dir):
        super().__init__()
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        last_path = os.path.join(self.checkpoint_dir, 'last.tmp.keras')
        final_path = os.path.join(self.checkpoint_dir, 'last.keras')
        self.model.save(last_path)
        os.replace(last_path, final_path)
        info = {'completed_epoch': int(epoch + 1)}
        temp_info = os.path.join(self.checkpoint_dir, 'resume.tmp.json')
        final_info = os.path.join(self.checkpoint_dir, 'resume.json')
        with open(temp_info, 'w') as f:
            json.dump(info, f)
        os.replace(temp_info, final_info)


def fit_with_resume(model, run_name, train_x, train_y, val_x, val_y,
                    epochs=100, batch_size=4):
    """Resume from the last completed epoch; also keep best val_accuracy and val_loss."""
    checkpoint_dir = os.path.join(RESULT_DIR, run_name)
    os.makedirs(checkpoint_dir, exist_ok=True)
    last_model = os.path.join(checkpoint_dir, 'last.keras')
    resume_file = os.path.join(checkpoint_dir, 'resume.json')
    csv_file = os.path.join(checkpoint_dir, 'epoch_history.csv')

    initial_epoch = 0
    if os.path.exists(last_model) and os.path.exists(resume_file):
        with open(resume_file, 'r') as f:
            initial_epoch = int(json.load(f).get('completed_epoch', 0))
        model = tf.keras.models.load_model(last_model)
        print(f'Resuming {run_name} from epoch {initial_epoch + 1}.')
    else:
        print(f'Starting {run_name} from epoch 1.')

    callbacks = [
        ModelCheckpoint(os.path.join(checkpoint_dir, 'best_val_accuracy.keras'),
                        monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
        ModelCheckpoint(os.path.join(checkpoint_dir, 'best_val_loss.keras'),
                        monitor='val_loss', mode='min', save_best_only=True, verbose=1),
        CSVLogger(csv_file, append=os.path.exists(csv_file)),
        SaveLastEpoch(checkpoint_dir)  # Save last, after other epoch-end callbacks.
    ]

    if initial_epoch < epochs:
        model.fit(train_x, train_y, validation_data=(val_x, val_y),
                  epochs=epochs, initial_epoch=initial_epoch,
                  batch_size=batch_size, callbacks=callbacks, verbose=1)
    else:
        print(f'{run_name} already reached epoch {epochs}; loading saved results.')

    # Keep last.keras for continuation, but return the best validation-accuracy model for evaluation.
    best_model_path = os.path.join(checkpoint_dir, 'best_val_accuracy.keras')
    if os.path.exists(best_model_path):
        model = tf.keras.models.load_model(best_model_path)
    elif os.path.exists(last_model):
        model = tf.keras.models.load_model(last_model)
    if os.path.exists(csv_file):
        history_df = pd.read_csv(csv_file)
        history = type('HistoryFromCSV', (), {'history': {
            col: history_df[col].tolist() for col in history_df.columns
            if col != 'epoch'
        }})()
    else:
        history = type('HistoryFromCSV', (), {'history': {}})()
    return model, history

print('Drive result folder:', RESULT_DIR)
print('Each completed epoch saves last.keras + resume.json. Best validation checkpoints are saved separately.')


#Optimizer: The optimizer is responsible for updating the weights of the neural network during training in order to minimize the loss function.

'adam': Adam (short for Adaptive Moment Estimation) is an efficient optimization algorithm that combines the advantages of two other extensions of stochastic gradient descent—AdaGrad and RMSProp. **Adam makes models learn more efficiently by continuously adjusting the learning rate of each parameter and, as a consequence, tends to converge much more quickly than standard stochastic gradient descent. **

Loss Function: The loss function measures how well the model's predictions match the true labels. During training, the optimizer adjusts the weights to minimize this loss.

'categorical_crossentropy': This loss function is commonly used for multi-class classification problems. It calculates the difference between the predicted probability distribution and the true distribution (one-hot encoded labels). The goal is to minimize this difference, thus improving the model's accuracy.

Metrics: Metrics are used to evaluate the performance of the model. Unlike the loss function, which the optimizer uses to train the model, metrics are for human interpretation and understanding of the model's performance.

['accuracy']: By including 'accuracy' as a metric, the model will calculate and display the accuracy of the predictions during training and evaluation. Accuracy is the ratio of correctly predicted instances to the total instances.

#Using validation

In [ ]:
# Train custom VGG19. If interrupted, run this cell again to resume automatically.
model, history = fit_with_resume(
    model, 'custom_vgg19', train_x, train_y, val_x, val_y,
    epochs=100, batch_size=4
)


In [ ]:
# Do not train on the test set. The custom VGG19 training and checkpoints are handled above.
# Keep test_x/test_y untouched for final testing.
print('Custom VGG19 training history is loaded from Drive. Test data is reserved for final evaluation.')


In [ ]:
# history= model.fit(train_x, train_y, epochs = 50, batch_size = 4, validation_data=(test_x, test_y))

In [ ]:
model.save('my_model.keras')  # save a trained Keras model to a file. This allows you to store the model's architecture, weights, and optimizer state so that it can be loaded later for inference, further training, or sharing with others.

#Model overview

In [ ]:
history.history['accuracy'][-1] #ccess the final training accuracy of the model after training.
#The history object contains a history attribute, which is a dictionary storing the metrics (e.g., loss, accuracy) recorded during training for each epoch.

In [ ]:
plt.plot(history.history['accuracy'],label='Accuracy')
plt.plot(history.history['val_accuracy'],label='Val_Accuracy')
plt.legend();
plt.savefig(os.path.join(RESULT_DIR, 'custom_vgg19_accuracy.png'), dpi=300, bbox_inches='tight')

In [ ]:
evaluation = model.evaluate(val_x, val_y)  #The evaluate method returns a list of values, which is stored in the variable evaluation.
print("Validation Loss:", evaluation[0])  #Accesses the loss value (first element in the list).
print("Validation Accuracy:", evaluation[1])
with open(os.path.join(RESULT_DIR, "custom_vgg19_validation_metrics.json"), "w") as f:
    json.dump({"validation_loss": float(evaluation[0]), "validation_accuracy": float(evaluation[1])}, f, indent=4)
#Evaluate the model's performance on unseen data (validation or test set).
#This helps you understand if the model has generalized well or if it is overfitting (memorizing the training data).


#Testing the Model

In [ ]:
base_model = VGG19(weights='imagenet', include_top = False, input_shape = (224, 224, 3)) #loaded pre trained VGG19 model on the ImageNet dataset, Loads the pre-trained weights from the ImageNet dataset.
#include_top = False Excludes the fully connected (dense) layers at the top of the network. This allows you to add your own custom layers for your specific task.
#weights are the learnable parameters of the model that determine how input data is transformed into output predictions
#If your task is to classify images into one of the 1,000 ImageNet classes, you can use include_top=True.


for layer in base_model.layers:
    layer.trainable = True          #  make all layers of a pre-trained model trainable
#This loop sets all layers of the base model to be trainable. By default, when loading a pre-trained model, the layers are often frozen (set to not be trainable). Making them trainable allows the model to fine-tune the weights during training on your specific dataset.
x = Flatten()(base_model.output)
x = Dense(1024, activation = 'relu')(x)
x = Dense(5, activation = 'softmax')(x) #This adds the output layer with 5 neurons (for 5 classes) and a softmax activation function to produce class probabilities.

model_post = Model(inputs = base_model.input, outputs = x)
model_post.compile(optimizer=Adam(), loss = 'categorical_crossentropy', metrics = ['accuracy'])

#for layer in base_model.layers

In [ ]:
# for layer in base_model.layers:
    layer.trainable = True

#base_model.layers:

Refers to all the layers in the base_model, which is typically a pre-trained model like VGG19, ResNet, etc.

layer.trainable = True:

Sets the trainable attribute of each layer to True, meaning the weights of these layers will be updated during training.

In [ ]:
# Train pretrained VGG19. This is a separate model and has its own Drive checkpoints.
# If interrupted, run this cell again; it resumes from its last completed epoch.
model_post, base_history = fit_with_resume(
    model_post, 'pretrained_vgg19', train_x, train_y, val_x, val_y,
    epochs=100, batch_size=6
)


In [ ]:
plt.plot(base_history.history['accuracy'],label='Accuracy')
plt.plot(base_history.history['val_accuracy'],label='Val_Accuracy')
plt.legend();
plt.savefig(os.path.join(RESULT_DIR, 'pretrained_vgg19_accuracy.png'), dpi=300, bbox_inches='tight')

In [ ]:
evaluation_x = model_post.evaluate(test_x, test_y)
print("Validation Loss:", evaluation_x[0])
print("Validation Accuracy:", evaluation_x[1])
with open(os.path.join(RESULT_DIR, "pretrained_vgg19_test_metrics.json"), "w") as f:
    json.dump({"test_loss": float(evaluation_x[0]), "test_accuracy": float(evaluation_x[1])}, f, indent=4)


In [ ]:
custom_test_eval = model.evaluate(test_x, test_y, verbose=0)
print("Custom VGG19 Test Loss:", custom_test_eval[0])
print("Custom VGG19 Test Accuracy:", custom_test_eval[1])
with open(os.path.join(RESULT_DIR, "custom_vgg19_test_metrics.json"), "w") as f:
    json.dump({"test_loss": float(custom_test_eval[0]), "test_accuracy": float(custom_test_eval[1])}, f, indent=4)


In [ ]:
y_pred1 = model.predict(test_x) #This is likely your initial model, perhaps using the original architecture or configuration before making changes or fine-tuning.
y_pred2 = model_post.predict(test_x) #his appears to be a modified version of the initial model, where you added custom layers and fine-tuned the pre-trained VGG19 model.
np.savez(os.path.join(RESULT_DIR, "test_predictions_both_models.npz"),
         y_true=np.argmax(test_y, axis=1),
         custom_predictions=np.argmax(y_pred1, axis=1),
         pretrained_predictions=np.argmax(y_pred2, axis=1))


In [ ]:
print(y_pred2)
print(test_y)
print(len(test_y))

#classification report

In [ ]:
y_pred_classes1 = np.argmax(y_pred1, axis=1) #finds the index of the maximum probability for each sample, axis=1: This indicates that we want to find the maximum value along the second axis
print(y_pred_classes1)
y_true1 = np.argmax(test_y, axis=1) #Converts the true one-hot encoded labels (test_y) into class labels. it finds the index of the true class labels in test_y..test_y te one-hot encoded labels ache
print(y_true1)
report = classification_report(y_true1, y_pred_classes1, target_names = lists) #Generates a classification report to evaluate the model's performance

print(report)
with open(os.path.join(RESULT_DIR, "custom_vgg19_classification_report.txt"), "w") as f:
    f.write(report)

#precision- For Nazli, precision is 0.20, meaning 20% of the predictions for this class are correct.

recall- For Nazli, precision is 0.20, meaning 20% of the predictions for this class are correct.

The model is biased toward class 3 (Nazli) and predicts this class for all samples.

Support: The number of actual occurrences of each class in the dataset.

In [ ]:
y_pred_classes2 = np.argmax(y_pred2, axis=1) #finds the index of the maximum probability for each sample, axis=1: This indicates that we want to find the maximum value along the second axis
print(y_pred_classes2)
y_true2 = np.argmax(test_y, axis=1)
print(y_true2)

report = classification_report(y_true2, y_pred_classes2, target_names = lists) #Generates a classification report to evaluate the model's performance

print(report)
with open(os.path.join(RESULT_DIR, "pretrained_vgg19_classification_report.txt"), "w") as f:
    f.write(report)

#Overall, the model has an accuracy of 25%, indicating that there is significant room for improvement.

Ak: Precision of 0.21, Of all the samples predicted to be Ak, 21% were actually Ak; Recall of 0.30,Of all the actual Ak samples, 30% were correctly predicted as Ak; F1-score of 0.25

Ala_Idris: Precision of 0.67,Of all the samples predicted to be Ala_Idris, 67% were actually Ala_Idris; Recall of 0.10, F1-score of 0.17

Buzgulu: Precision of 0.26, Recall of 0.85,Of all the actual Buzgulu samples, 85% were correctly predicted as Buzgulu; F1-score of 0.40

Nazli: Precision, Recall, F1-score all zero because there were no correctly predicted samples of this class.

Dimnit: Precision, Recall, F1-score all zero because there were no correctly predicted samples of this class.

#Class Buzgulu has the highest recall (0.85), meaning that most of the true instances of Buzgulu were correctly predicted, but it still has relatively low precision (0.26), indicating that there were many false positives.

Class Ala_Idris has the highest precision (0.67), meaning that when the model predicts Ala_Idris, it is often correct. However, its recall is very low (0.10), meaning that most Ala_Idris instances are not being correctly identified.

Classes Nazli and Dimnit have both precision and recall of 0.00, indicating that the model failed to predict any instances of these classes correctly.

Class Ak has a moderate balance between precision (0.21) and recall (0.30), leading to a F1-score of 0.25, but still shows a significant need for improvement.

The overall accuracy of the model is 25%, which indicates that the model correctly predicts the class labels for 25% of the samples.

#Confusion Matrix

In [ ]:
import seaborn as sns #A data visualization library used for creating attractive and informative statistical graphics.
from sklearn.metrics import confusion_matrix #A function from sklearn.metrics that computes the confusion matrix to evaluate the accuracy of a classification.

cm1 = confusion_matrix(y_true1, y_pred_classes1)
cm2 = confusion_matrix(y_true2, y_pred_classes2)
np.savetxt(os.path.join(RESULT_DIR, "custom_vgg19_confusion_matrix.csv"), cm1, delimiter=",", fmt="%d")
np.savetxt(os.path.join(RESULT_DIR, "pretrained_vgg19_confusion_matrix.csv"), cm2, delimiter=",", fmt="%d")

plt.figure(figsize=(20, 5))
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', xticklabels=lists, yticklabels=lists)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix custom')
plt.savefig(os.path.join(RESULT_DIR, 'custom_vgg19_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(20, 5))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Reds', xticklabels=lists, yticklabels=lists)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix Pre Trained')
plt.savefig(os.path.join(RESULT_DIR, 'pretrained_vgg19_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

#Diagonal Values (Correct Predictions):

These values indicate correct classifications (where the actual class matches the predicted class).

Ak: 6 correctly classified.

Ala_Idris: 2 correctly classified.

Buzgulu: 17 correctly classified.

Nazli: 0 correctly classified.

Dimnit: 0 correctly classified.

The model performs well for Buzgulu but fails completely on Nazli and Dimnit.

#Misclassifications (Off-Diagonal Entries):

Ak is often misclassified as Buzgulu (14 times).

Ala_Idris is misclassified as Buzgulu (9 times) and sometimes as Dimnit (3 times).

Nazli is heavily misclassified as Buzgulu (15 times).

Dimnit is often misclassified as Ak (8 times) and Buzgulu (11 times).

#General Observations:

The model seems biased towards predicting Buzgulu (frequent misclassification into this category).

Nazli and Dimnit are never correctly classified, indicating poor performance for these classes.

In [ ]:
plt.figure(figsize=(20, 5))
plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'],  label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Testing Accuracy')
plt.plot(history.history['loss'], label='Loss')
plt.title('Training and Tranig Accuracy loss on Custom VGG19')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot training and validation loss
plt.subplot(1, 3, 2)
plt.plot(base_history.history['accuracy'],  label='Training Accuracy')
plt.plot(base_history.history['val_accuracy'], label='Testing Accuracy')
plt.plot(base_history.history['loss'], label='Loss')
plt.title('Training and Tranig Accuracy, Loss on Pre VGG19')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'training_curves_both_models.png'), dpi=300, bbox_inches='tight')
plt.show()

#Drive checkpoint and result location
All training checkpoints and result files are saved automatically in MyDrive/Grapevine_VGG19_Results.
- custom_vgg19/last.keras: latest completed epoch; used to resume training.
- custom_vgg19/best_val_accuracy.keras: best validation-accuracy model.
- custom_vgg19/best_val_loss.keras: best validation-loss model.
- Equivalent files are saved under pretrained_vgg19/.
- Epoch CSV logs, validation metrics, classification reports, confusion matrices, and plots are also saved in the same Drive folder.
Resume rule: if Colab disconnects, reconnect Drive and run the notebook cells in order up to the training cell, then run that training cell again. It resumes from the last fully completed epoch, not from epoch 1. If interruption happens in the middle of an epoch, that incomplete epoch is repeated because only completed epochs are checkpointed.